# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency: A Clinical Analysis and Literature Review Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
This dataset is defined via a Croissant schema at:
https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json

The dataset includes tabular records for clinical features, hormone levels, adrenal imaging, and genetic variants for three female patients diagnosed with non-classical 21-hydroxylase deficiency-associated infertility.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
pprint.pprint(metadata.to_json())

print(f"Dataset name: {metadata.name}")
print(f"Description: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

We enumerate all record sets in the dataset, listing their `@id`s and associated fields. All subsequent references to dataset entities use their `@id` for clarity and reproducibility.

In [ ]:
# List all record sets with their @id and fields
record_sets = metadata.recordSets

print("Record Sets:")
for rs in record_sets:
    print(f"  Record Set: {rs['@id']}")
    print(f"    Name: {rs.get('name', 'N/A')}")
    print(f"    Description: {rs.get('description', 'N/A')}")
    fields = rs.get('fields', [])
    print(f"    Fields:")
    for f in fields:
        print(f"      - Field @id: {f['@id']}, Name: {f.get('name', 'N/A')}")

## 3. Data Extraction
Load data from each record set into DataFrames for analysis.

Below, we use the record set and field `@id`s from the overview. For demonstration, we extract all available record sets in the FAIR^2 dataset.

In [ ]:
# Extract data for each record set (@id)
# Use the @id from the record_sets variable
dataframes = {}
rs_ids = [rs['@id'] for rs in metadata.recordSets]
print("Extracting data for record sets:", rs_ids)

for rs_id in rs_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"\nRecord Set {rs_id} columns: {df.columns.tolist()}")
    print(df.head(3))

## 4. Exploratory Data Analysis (EDA)
Apply common processing steps, such as filtering records on a criterion, normalizing numeric columns, categorizing, outlier removal, etc. using only `@id` references.

For illustration, let's:
- Select the "clinical features" record set (Table 1),
- Filter patients by age,
- Normalize the age column,
- Group by a categorical field such as phenotype or clinical status, if present.

In [ ]:
# Pick a clinical features record set and field by @id
# Replace below with actual @ids from your record set overview.

# Example: Table 1 record set
clinical_rs_id = rs_ids[0]  # Replace with exact @id if needed
df = dataframes[clinical_rs_id]

# Find numeric field (e.g., age)
numeric_field_id = None
for f in [rs for rs in metadata.recordSets if rs['@id'] == clinical_rs_id][0]['fields']:
    # Pick 'age' field by name or @id
    if 'age' in f.get('name', '').lower():
        numeric_field_id = f['@id']
        break

# If not found, print first numeric column
if not numeric_field_id:
    numeric_cols = df.select_dtypes(include='number').columns.tolist()
    if numeric_cols:
        numeric_field_id = numeric_cols[0]

print(f"Numeric field selected for analysis: {numeric_field_id}")

threshold = 10  # e.g., age > 10
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold}:")
print(filtered_df.head())

filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Try grouping by a categorical field if present (e.g., phenotype or clinical status)
group_field_id = None
for f in [rs for rs in metadata.recordSets if rs['@id'] == clinical_rs_id][0]['fields']:
    # Example: 'phenotype' or string clinical field
    if f.get('dataType') == 'Text' and f.get('name','').lower() not in ['age']:
        group_field_id = f['@id']
        break
if group_field_id and group_field_id in filtered_df.columns:
    grouped = filtered_df.groupby(group_field_id).mean(numeric_only=True)
    print(f"Grouped by {group_field_id}:")
    print(grouped.head())

## 5. Visualization
Visualize distributions and relationships using numeric and categorical fields.

Below, we plot a histogram of the selected numeric field (age), and scatter plot if a second numeric field is available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of normalized age
plt.figure(figsize=(6,4))
sns.histplot(filtered_df[f"{numeric_field_id}_normalized"], bins=5, kde=True)
plt.title(f"Normalized {numeric_field_id} Distribution")
plt.xlabel("Normalized Value")
plt.ylabel("Frequency")
plt.show()

# If a second numeric field exists, plot scatter
second_numeric = [col for col in df.select_dtypes(include='number').columns if col != numeric_field_id]
if second_numeric:
    second_numeric_id = second_numeric[0]
    plt.figure(figsize=(6,4))
    sns.scatterplot(x=filtered_df[numeric_field_id], y=filtered_df[second_numeric_id])
    plt.title(f"{numeric_field_id} vs {second_numeric_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel(second_numeric_id)
    plt.show()

## 6. Conclusion

We have:
- Loaded FAIR^2 dataset metadata and records using the Croissant schema and `mlcroissant`.
- Used `@id` references for record sets and fields for reproducibility.
- Performed basic EDA: filtering, normalization, grouping, and visualizations.

Key findings:
- The clinical features record set enables filtering and normalization for patient-level analysis.
- Given the very small sample (N=3), distributions are illustrative and should be interpreted cautiously.
- The use of Croissant schemas and `mlcroissant` facilitates structured, reproducible exploration for research datasets.

**Note:** Due to clinical privacy, sensitive fields (such as gender, age) reference protected attributes. Further exploration should respect data governance and limitations outlined in the metadata.

<!-- End of notebook -->